# Safe Malware Static-Feature Classification

Classify synthetic PE-like metadata without downloading, opening, or executing binaries.

**Safety and scope:** This project uses synthetic, non-sensitive telemetry for defensive analytics. It does not perform exploitation or execute malicious content.

## Goal

Train an explainable classifier on static metadata and inspect the features associated with suspicious samples.


## Setup

The notebook is deterministic, runs offline, and implements the core analytical method directly with NumPy and Pandas so the modeling logic remains inspectable.


In [1]:
import numpy as np
import pandas as pd

SEED = 42
rng = np.random.default_rng(SEED)
pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 120)

def sigmoid(values):
    clipped = np.clip(values, -30, 30)
    return 1.0 / (1.0 + np.exp(-clipped))

def split_indices(size, test_fraction=0.25):
    shuffled = rng.permutation(size)
    split_at = int(size * (1 - test_fraction))
    return shuffled[:split_at], shuffled[split_at:]

def standardize(train_values, test_values):
    mean = train_values.mean(axis=0)
    std = train_values.std(axis=0)
    std = np.where(std < 1e-9, 1.0, std)
    return (train_values - mean) / std, (test_values - mean) / std, mean, std

def fit_logistic(features, labels, steps=1400, learning_rate=0.08, l2=0.01):
    design = np.column_stack([np.ones(len(features)), features])
    weights = np.zeros(design.shape[1])
    for _ in range(steps):
        probabilities = sigmoid(design @ weights)
        gradient = design.T @ (probabilities - labels) / len(labels)
        gradient[1:] += l2 * weights[1:]
        weights -= learning_rate * gradient
    return weights

def predict_probability(features, weights):
    design = np.column_stack([np.ones(len(features)), features])
    return sigmoid(design @ weights)

def classification_metrics(labels, predictions):
    labels = np.asarray(labels)
    predictions = np.asarray(predictions)
    tp = int(((labels == 1) & (predictions == 1)).sum())
    tn = int(((labels == 0) & (predictions == 0)).sum())
    fp = int(((labels == 0) & (predictions == 1)).sum())
    fn = int(((labels == 1) & (predictions == 0)).sum())
    precision = tp / max(tp + fp, 1)
    recall = tp / max(tp + fn, 1)
    f1 = 2 * precision * recall / max(precision + recall, 1e-12)
    accuracy = (tp + tn) / max(len(labels), 1)
    return pd.Series({
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "tp": tp,
        "fp": fp,
        "tn": tn,
        "fn": fn,
    })


## Steps

### 1. Generate synthetic static metadata


In [2]:
sample_count = 1700
file_entropy = rng.normal(5.3, 0.9, sample_count).clip(1.0, 8.0)
section_count = rng.poisson(4.5, sample_count).clip(1, 12)
suspicious_imports = rng.poisson(1.1, sample_count)
signed_binary = rng.binomial(1, 0.64, sample_count)
packer_hint = rng.binomial(1, 0.12, sample_count)
executable_writable_sections = rng.binomial(1, 0.10, sample_count)
string_count_log = rng.normal(7.0, 0.8, sample_count).clip(3.0, 10.0)

malicious_probability = sigmoid(
    -2.5
    + 0.75 * (file_entropy - 5)
    + 0.32 * suspicious_imports
    - 1.05 * signed_binary
    + 1.85 * packer_hint
    + 1.55 * executable_writable_sections
    - 0.18 * (string_count_log - 7)
)
malicious = rng.binomial(1, malicious_probability)

static_features = pd.DataFrame({
    "file_entropy": file_entropy,
    "section_count": section_count,
    "suspicious_imports": suspicious_imports,
    "signed_binary": signed_binary,
    "packer_hint": packer_hint,
    "executable_writable_sections": executable_writable_sections,
    "string_count_log": string_count_log,
    "malicious": malicious,
})

print("Sample count:", len(static_features))
print("Synthetic malicious rate:", round(static_features["malicious"].mean(), 3))
print(static_features.head(6).round(3).to_string(index=False))


Sample count: 1700
Synthetic malicious rate: 0.141
 file_entropy  section_count  suspicious_imports  signed_binary  packer_hint  executable_writable_sections  string_count_log  malicious
        5.574              8                   2              0            0                             0             7.429          0
        4.364              8                   1              0            1                             0             7.051          0
        5.975              2                   0              1            1                             0             5.787          1
        6.147              1                   1              1            0                             0             7.204          0
        3.544              2                   1              1            0                             0             6.490          0
        4.128              9                   1              1            0                             1             5.788         

### 2. Train and explain the classifier


In [3]:
feature_names = [column for column in static_features.columns if column != "malicious"]
train_rows, test_rows = split_indices(len(static_features))
train_values = static_features.loc[train_rows, feature_names].to_numpy(float)
test_values = static_features.loc[test_rows, feature_names].to_numpy(float)
train_labels = static_features.loc[train_rows, "malicious"].to_numpy(int)
test_labels = static_features.loc[test_rows, "malicious"].to_numpy(int)

train_scaled, test_scaled, _, _ = standardize(train_values, test_values)
malware_weights = fit_logistic(train_scaled, train_labels)
malware_probability = predict_probability(test_scaled, malware_weights)
malware_threshold = 0.20
malware_prediction = (malware_probability >= malware_threshold).astype(int)
malware_metrics = classification_metrics(test_labels, malware_prediction)

malware_importance = pd.DataFrame({
    "feature": feature_names,
    "standardized_weight": malware_weights[1:],
}).sort_values("standardized_weight", ascending=False)
ranked_samples = static_features.loc[test_rows].copy()
ranked_samples["malicious_probability"] = malware_probability
ranked_samples = ranked_samples.sort_values("malicious_probability", ascending=False)

print("Test metrics:")
print(malware_metrics.round(3).to_string())
print("\nFeature weights:")
print(malware_importance.round(3).to_string(index=False))
print("\nHighest-risk metadata samples:")
print(ranked_samples.head(8).round(3).to_string(index=False))


Test metrics:
accuracy       0.852
precision      0.489
recall         0.721
f1             0.583
tp            44.000
fp            46.000
tn           318.000
fn            17.000

Feature weights:
                     feature  standardized_weight
                file_entropy                0.605
                 packer_hint                0.480
executable_writable_sections                0.448
          suspicious_imports                0.397
               section_count                0.024
            string_count_log               -0.131
               signed_binary               -0.362

Highest-risk metadata samples:
 file_entropy  section_count  suspicious_imports  signed_binary  packer_hint  executable_writable_sections  string_count_log  malicious  malicious_probability
        6.319              8                   2              0            1                             1             6.658          1                  0.885
        5.977              4                   0  

## Checks


In [4]:
assert static_features["file_entropy"].between(1, 8).all()
assert malware_metrics["f1"] >= 0.45
assert ranked_samples["malicious_probability"].is_monotonic_decreasing
assert malware_importance.iloc[0]["standardized_weight"] > 0
print("Checks passed: bounded metadata, usable synthetic-data F1, sorted risk, and positive risk drivers.")


Checks passed: bounded metadata, usable synthetic-data F1, sorted risk, and positive risk drivers.


## Next Steps

        - Use an approved static-analysis pipeline and never execute unknown samples in this notebook.
- Evaluate family and time-based splits to expose concept drift.
- Combine static signals with sandbox results only in an isolated malware-analysis environment.
